# Session 8 Part 1: PySpark in Google Colab

In [ ]:
!pip install pyspark==4.1.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079457 sha256=ed5f5b5447fd4af7369b128870293a143e0039ad8b227732de23ff877c145760
  Stored in directory: /root/.cache/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.1.2 which is incompatible.


In [ ]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Session08Part01")
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
students = [
    ("Ava", "Data Analytics", 92),
    ("Nikos", "Data Engineering", 85),
    ("Maya", "Data Analytics", 78),
    ("Leo", "Business", 88),
    ("Iris", "Data Engineering", 95),
]

columns = ["name", "track", "score"]

df = spark.createDataFrame(students, columns)

df.show()

+-----+----------------+-----+
| name|           track|score|
+-----+----------------+-----+
|  Ava|  Data Analytics|   92|
|Nikos|Data Engineering|   85|
| Maya|  Data Analytics|   78|
|  Leo|        Business|   88|
| Iris|Data Engineering|   95|
+-----+----------------+-----+



Spark prints a table with name, track, and score.

In [ ]:
students = [
    ("Ava", "Data Analytics", 92),
    ("Nikos", "Data Engineering", 85),
    ("Maya", "Data Analytics", 78),
    ("Leo", "Business", 88),
    ("Iris", "Data Engineering", 95),
    ("Elena", "Business", 81),
    ("Omar", "Data Analytics", 90),
]

df = spark.createDataFrame(students, columns)

df.show()

+-----+----------------+-----+
| name|           track|score|
+-----+----------------+-----+
|  Ava|  Data Analytics|   92|
|Nikos|Data Engineering|   85|
| Maya|  Data Analytics|   78|
|  Leo|        Business|   88|
| Iris|Data Engineering|   95|
|Elena|        Business|   81|
| Omar|  Data Analytics|   90|
+-----+----------------+-----+



7. Inspect the DataFrame

In [ ]:
df.printSchema()
print("Rows:", df.count())
print("Columns:", df.columns)

root
 |-- name: string (nullable = true)
 |-- track: string (nullable = true)
 |-- score: long (nullable = true)

Rows: 7
Columns: ['name', 'track', 'score']


What this shows:

printSchema() shows column names and inferred data types.

1.   printSchema() shows column names and inferred data types.
2.   count() counts rows.
3.   columns lists the column names.


Checkpoint question:

Which command is an action: printSchema, count, or columns?

count() is the clearest action here because it asks Spark to compute a result from the data.

Checkpoint question:
Why is checking the schema useful before writing filters or calculations?

The schema tells you which columns exist and what data type Spark thinks each column contains. This helps you avoid mistakes such as doing numeric comparisons on text columns.

# 8. Select columns

In [ ]:
df.select("name", "score").show()

+-----+-----+
| name|score|
+-----+-----+
|  Ava|   92|
|Nikos|   85|
| Maya|   78|
|  Leo|   88|
| Iris|   95|
|Elena|   81|
| Omar|   90|
+-----+-----+



Checkpoint question:

Does select("name", "score") change the original df?

No. Spark DataFrame operations return a new DataFrame. The original df still has all of its columns unless you assign the selected result to a variable.

Task: Show only the name and track columns.

In [ ]:
df.select("name", "track").show()

+-----+----------------+
| name|           track|
+-----+----------------+
|  Ava|  Data Analytics|
|Nikos|Data Engineering|
| Maya|  Data Analytics|
|  Leo|        Business|
| Iris|Data Engineering|
|Elena|        Business|
| Omar|  Data Analytics|
+-----+----------------+



# 9. Select with expressions and conditions

select can do more than choose existing columns. It can also show calculated expressions.

In [ ]:
from pyspark.sql.functions import col, when

df.select(
    "name",
    "score",
    (col("score") + 5).alias("score_plus_5")
).show()

+-----+-----+------------+
| name|score|score_plus_5|
+-----+-----+------------+
|  Ava|   92|          97|
|Nikos|   85|          90|
| Maya|   78|          83|
|  Leo|   88|          93|
| Iris|   95|         100|
|Elena|   81|          86|
| Omar|   90|          95|
+-----+-----+------------+



Checkpoint question:

Why do we use alias("score_plus_5") here?

The expression col("score") + 5 would otherwise have a long generated name. alias gives the calculated column a clear name in the output.

In [ ]:
df.select("name", "score").show()

+-----+-----+
| name|score|
+-----+-----+
|  Ava|   92|
|Nikos|   85|
| Maya|   78|
|  Leo|   88|
| Iris|   95|
|Elena|   81|
| Omar|   90|
+-----+-----+



You can also use a condition inside select:

In [ ]:
df.select(
    "name",
    "track",
    "score",
    when(col("score") >= 85, "pass").otherwise("review").alias("result")
).show()

+-----+----------------+-----+------+
| name|           track|score|result|
+-----+----------------+-----+------+
|  Ava|  Data Analytics|   92|  pass|
|Nikos|Data Engineering|   85|  pass|
| Maya|  Data Analytics|   78|review|
|  Leo|        Business|   88|  pass|
| Iris|Data Engineering|   95|  pass|
|Elena|        Business|   81|review|
| Omar|  Data Analytics|   90|  pass|
+-----+----------------+-----+------+



What this shows:



1.   when(...) checks a condition.
2.   otherwise(...) gives the value when the condition is false.
3.   alias("result") names the new output column.




Checkpoint question:

Does this conditional select permanently add the result column to df?

No. It only shows a selected output with a calculated column. If you want to keep the new column, use withColumn and assign the result to a variable.

Task: Select name, score, and a new column named score_level.

Use:

1.   "excellent" when the score is at least 90
2.   "good" when the score is at least 80
2.   "needs review" for all other scores

In [ ]:
df.select(
    "name",
    "score",
    when(col("score") >= 90, "excellent")
    .when(col("score") >= 80, "good")
    .otherwise("needs review")
    .alias("score_level")
).show()

+-----+-----+------------+
| name|score| score_level|
+-----+-----+------------+
|  Ava|   92|   excellent|
|Nikos|   85|        good|
| Maya|   78|needs review|
|  Leo|   88|        good|
| Iris|   95|   excellent|
|Elena|   81|        good|
| Omar|   90|   excellent|
+-----+-----+------------+



# 10. Filter rows

Use filter to keep rows that match a condition:

In [ ]:
df.filter(df["score"] >= 90).show()

+----+----------------+-----+
|name|           track|score|
+----+----------------+-----+
| Ava|  Data Analytics|   92|
|Iris|Data Engineering|   95|
|Omar|  Data Analytics|   90|
+----+----------------+-----+



Checkpoint question:

What rows should this filter keep?

Show answer
It should keep only students whose score is greater than or equal to 90.

Task: Show students whose score is less than 85.

In [ ]:
df.filter(df["score"] < 85).show()

+-----+--------------+-----+
| name|         track|score|
+-----+--------------+-----+
| Maya|Data Analytics|   78|
|Elena|      Business|   81|
+-----+--------------+-----+



show students with name Maya

In [ ]:
df.filter(df["name"] == "Maya").show()

+----+--------------+-----+
|name|         track|score|
+----+--------------+-----+
|Maya|Data Analytics|   78|
+----+--------------+-----+



What is the difference between this code and df.select("name", "track", "score")?

Show answer
df.select("name", "track", "score") shows those columns for every row. df.filter(col("score") >= 85).select(...) first keeps only high-score rows, then shows the selected columns.

Task: Show only the name and score columns for students in the Data Analytics track.

In [ ]:
df.filter(df["track"] == "Data Analytics").select("name", "score").show()

+----+-----+
|name|score|
+----+-----+
| Ava|   92|
|Maya|   78|
|Omar|   90|
+----+-----+



# 11. Create a new column

Import col and when:

In [ ]:
from pyspark.sql.functions import col, when

graded_df = df.withColumn(
    "result",
    when(col("score") >= 85, "pass").otherwise("review")
)

graded_df.show()

+-----+----------------+-----+------+
| name|           track|score|result|
+-----+----------------+-----+------+
|  Ava|  Data Analytics|   92|  pass|
|Nikos|Data Engineering|   85|  pass|
| Maya|  Data Analytics|   78|review|
|  Leo|        Business|   88|  pass|
| Iris|Data Engineering|   95|  pass|
|Elena|        Business|   81|review|
| Omar|  Data Analytics|   90|  pass|
+-----+----------------+-----+------+



Checkpoint question:

Why do we save the result as graded_df instead of expecting df to change automatically?

Show answer
Spark DataFrames are immutable. withColumn returns a new DataFrame with the extra column, so assigning it to graded_df keeps the new version.

Task: Add a new column named score_plus_5 that adds 5 points to each score.

Show solution

In [ ]:
df.withColumn("score_plus_5", col("score") + 5).show()

+-----+----------------+-----+------------+
| name|           track|score|score_plus_5|
+-----+----------------+-----+------------+
|  Ava|  Data Analytics|   92|          97|
|Nikos|Data Engineering|   85|          90|
| Maya|  Data Analytics|   78|          83|
|  Leo|        Business|   88|          93|
| Iris|Data Engineering|   95|         100|
|Elena|        Business|   81|          86|
| Omar|  Data Analytics|   90|          95|
+-----+----------------+-----+------------+



# 12. Group and summarize

Use groupBy to summarize by category:

In [ ]:
df.groupBy("track").count().show()

+----------------+-----+
|           track|count|
+----------------+-----+
|Data Engineering|    2|
|  Data Analytics|    3|
|        Business|    2|
+----------------+-----+



Checkpoint question:

What does each row in this grouped result represent?

Show answer

Each row represents one track, plus the number of students in that track.

Now calculate average score by track:

the `agg` function is essential with `groupBy` because `groupBy` creates a `GroupedData` object, but it doesn't perform any calculations on its own. The `agg` function is then used to apply one or more aggregate functions (like `avg`, `sum`, `count`, `min`, `max`) to the grouped data.

In [ ]:
from pyspark.sql.functions import avg

df.groupBy("track").agg(avg("score").alias("average_score")).show()

+----------------+-----------------+
|           track|    average_score|
+----------------+-----------------+
|Data Engineering|             90.0|
|  Data Analytics|86.66666666666667|
|        Business|             84.5|
+----------------+-----------------+



Checkpoint question:

Why do we use alias("average_score")?

Show answer

Without an alias, Spark may show a generated column name such as avg(score). The alias gives the summary column a clean readable name.

Task: Find the maximum score in each track.

In [ ]:
from pyspark.sql.functions import max

df.groupBy("track").agg(max("score").alias("max_score")).show()

+----------------+---------+
|           track|max_score|
+----------------+---------+
|Data Engineering|       95|
|  Data Analytics|       92|
|        Business|       88|
+----------------+---------+



In [ ]:
df.groupBy("track").agg(max("score")).show()

+----------------+----------+
|           track|max(score)|
+----------------+----------+
|Data Engineering|        95|
|  Data Analytics|        92|
|        Business|        88|
+----------------+----------+



# 13. Exercise 1: PySpark student summary

Create a final Colab section called:

## Exercise 1

Your notebook should:

1.  Install PySpark.
2.  Start a Spark session.
3.  Create a DataFrame with at least 8 students.
    Include these columns:
    *   name
    *   track
    *   score
    *   hours_studied
4.  Print the schema.
5.  Show the first 5 rows.
6.  Filter students with score >= 85.
7.  Add a result column with pass or review.
8.  Group by track and show:
    *   number of students
    *   average score
    *   average hours studied
9.  Stop Spark at the end.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, when


spark = (
    SparkSession.builder
    .appName("Session08Exercise01")
    .master("local[*]")
    .getOrCreate()
)

students = [
    # TODO: add at least 8 rows
    ("Ava", "Data Analytics", 92, 40),
    ("Nikos", "Data Engineering", 85, 50),
    ("Maya", "Data Analytics", 78, 45),
    ("Leo", "Business", 88, 55),
    ("Iris", "Data Engineering", 95, 60),
    ("Elena", "Business", 81, 50),
    ("Omar", "Data Analytics", 90, 67),
    ("Stelios", "Data Analytics", 88, 50),
    ("Dara", "Data Engineering", 89, 67),
]

columns = ["name", "track", "score", "hours_studied"]

df = spark.createDataFrame(students, columns)

# TODO: print schema
df.printSchema()

# TODO: show first 5 rows
df.show(5)

# TODO: filter high scores
df.filter(df["score"] >= 85).show()

# TODO: add result column
df.withColumn(
    "result",
    when(col("score") >= 85, "pass").otherwise("review")
).show()

# TODO: group by track
df.groupBy("track").agg(
    count("name").alias("number_of_students"),
    avg("score").alias("average_score"),
    avg("hours_studied").alias("average_hours_studied")
).show()

spark.stop()

root
 |-- name: string (nullable = true)
 |-- track: string (nullable = true)
 |-- score: long (nullable = true)
 |-- hours_studied: long (nullable = true)

+-----+----------------+-----+-------------+
| name|           track|score|hours_studied|
+-----+----------------+-----+-------------+
|  Ava|  Data Analytics|   92|           40|
|Nikos|Data Engineering|   85|           50|
| Maya|  Data Analytics|   78|           45|
|  Leo|        Business|   88|           55|
| Iris|Data Engineering|   95|           60|
+-----+----------------+-----+-------------+
only showing top 5 rows
+-------+----------------+-----+-------------+
|   name|           track|score|hours_studied|
+-------+----------------+-----+-------------+
|    Ava|  Data Analytics|   92|           40|
|  Nikos|Data Engineering|   85|           50|
|    Leo|        Business|   88|           55|
|   Iris|Data Engineering|   95|           60|
|   Omar|  Data Analytics|   90|           67|
|Stelios|  Data Analytics|   88|      